Loading the Libraries


In [4]:
import h5py
import numpy as np
import pandas as pd
import sklearn

Extracting the .h5 file Data 

In [5]:


# Replace with your actual path
file_path = "/nfs/home/users/vpandya/data/shared/1 Week/Kai-Rakovic-M42/Vaidehi/hdf5_TCGA_LUAD_5x_he_train_reps.h5"

with h5py.File(file_path, "r") as f:
    print("Keys in file:")
    print(list(f.keys()))

Keys in file:
['train_hist_subtype', 'train_img_h_latent', 'train_img_z_latent', 'train_labels', 'train_patterns', 'train_samples', 'train_slides', 'train_tiles']


Checking the z_Latent Embeddings

In [6]:


with h5py.File(file_path, "r") as f:
    data = np.array(f["train_img_z_latent"])
    
print("Shape:", data.shape)
print("Sample values:", data[:5])

Shape: (826977, 128)
Sample values: [[-0.5355677   1.9926075   1.9746926   1.4493992   0.35555443 -0.46787724
  -1.4886032   0.93662184 -1.1952565   0.81789595 -2.1286879  -1.9279262
  -0.69896185  2.3411129  -0.8922369   1.3257779  -1.8474056  -1.5694778
   0.0778975  -0.84290546  1.4481298   1.4650356  -0.97966874 -0.11160925
   0.0523894  -0.61649734 -0.04736368 -0.02519128  0.15564214 -0.52593863
  -0.34065133  0.33289522 -1.2377132  -1.4393531  -2.2551463   1.031163
  -0.77659535 -1.4247156  -0.07834993  0.9845995   0.9993402   0.9396653
  -0.41364956 -0.7533982   2.2075508  -0.9897113   0.80901945  1.204118
  -1.4625583   0.1886499  -1.5575833   0.7187794  -1.00006    -1.0005565
  -0.37060794  0.23825969  0.9904999   0.90733135 -3.2241204   1.2938052
  -0.22010618  0.7004942  -3.2670035   0.54204756  0.5547276   1.1281487
   1.4354451  -0.26751578  1.9682605  -0.23632434 -2.1844776  -1.2331352
  -1.1869215  -0.34988803 -0.43281305 -2.058033   -0.63029325  1.6966084
   0.85490716 

Comparring the rows in Z_latent and tiles and slides data from the HPL run

In [7]:



with h5py.File(file_path, "r") as f:
    z_latent = np.array(f["train_img_z_latent"])
    train_tiles = np.array(f["train_tiles"]).astype(str)
    train_slides =np.array(f["train_slides"]).astype(str)

print("Embeddings shape:", z_latent.shape)
print("Tile count:", len(train_tiles))
print("Slide count:",len(train_slides))

Embeddings shape: (826977, 128)
Tile count: 826977
Slide count: 826977


Merging Slide and tile data for slide_tile

In [8]:

with h5py.File(file_path, "r") as f:
 
    tile_names = f["train_tiles"][:]
    slide_names = f["train_slides"][:]

# decode byte strings if needed
tile_names = [t.decode("utf-8") if isinstance(t, bytes) else t for t in tile_names]
slide_names = [s.decode("utf-8") if isinstance(s, bytes) else s for s in slide_names]

slide_tile = [f"{s}_{t}" for s, t in zip(slide_names, tile_names)]
print("Unique slide_tile entries:", len(set(slide_tile)))
print("Total entries:", len(slide_tile))

Unique slide_tile entries: 826861
Total entries: 826977


Adding that to Z_latent

In [9]:
with h5py.File(file_path, "r") as f:
    z_latent = f["train_img_z_latent"][:]
z_cols = [f"z_{i}" for i in range(z_latent.shape[1])]
z_df = pd.DataFrame(z_latent, columns=z_cols)
z_df.insert(0, "slide_tile", slide_tile)

print(" z_df created successfully")
print("Shape:", z_df.shape)
display(z_df.head())

 z_df created successfully
Shape: (826977, 129)


,slide_tile,z_0,z_1,z_2,z_3,z_4,z_5,z_6,z_7,z_8,...,z_118,z_119,z_120,z_121,z_122,z_123,z_124,z_125,z_126,z_127
0,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,-0.535568,1.992607,1.974693,1.449399,0.355554,-0.467877,-1.488603,0.936622,-1.195256,...,1.027037,-1.529267,0.688359,1.131204,0.451487,1.851114,-1.451645,1.063994,0.495548,0.291344
1,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,-0.966781,0.355278,0.165919,-0.509434,1.690443,0.703921,-1.080820,-0.774081,1.289157,...,-0.977019,1.144525,-2.966762,-0.740570,1.469529,-0.962382,1.383525,-1.881596,-0.036612,-2.326094
2,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,0.942118,1.762963,-0.408902,0.514347,-1.126567,1.351461,1.762782,-0.871871,0.768488,...,-1.763825,0.624943,2.821767,1.070980,1.483262,-1.849699,-0.462854,-1.672111,1.660181,1.872878
3,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,-1.875595,0.050555,1.031914,-0.668931,-0.677426,-0.820803,-1.981195,-2.135660,-2.995363,...,-1.222338,-0.192205,-0.901741,1.838929,1.861430,-0.794674,2.664395,1.720787,-0.186903,-1.178385
4,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,-0.974751,2.498883,0.251445,1.301010,0.137949,-0.975547,0.428637,-0.868052,1.202727,...,-2.130194,-1.719221,-0.419598,2.851778,1.108923,-1.456366,-1.579241,0.228511,-0.281135,-1.521479


Saving as the merged z_latent data as .csv

In [10]:
z_df.to_csv("train_img_z_latent.csv.gz", index=False, compression="gzip")

print(" Saved the z_latent embeddings:")
print("  train_img_z_latent.csv.gz")


 Saved the z_latent embeddings:
  train_img_z_latent.csv.gz


Loading and checking the z_latent.csv

In [11]:
# Load the compressed z_latent file
z_latent = pd.read_csv("train_img_z_latent.csv.gz")

print("Loaded z_latent successfully!")
print("Shape:", z_latent.shape)
display(z_latent.head())

Loaded z_latent successfully!
Shape: (826977, 129)


,slide_tile,z_0,z_1,z_2,z_3,z_4,z_5,z_6,z_7,z_8,...,z_118,z_119,z_120,z_121,z_122,z_123,z_124,z_125,z_126,z_127
0,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,-0.535568,1.992608,1.974693,1.449399,0.355554,-0.467877,-1.488603,0.936622,-1.195256,...,1.027037,-1.529267,0.688359,1.131204,0.451487,1.851114,-1.451645,1.063994,0.495548,0.291344
1,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,-0.966781,0.355278,0.165919,-0.509434,1.690443,0.703921,-1.080820,-0.774081,1.289157,...,-0.977019,1.144525,-2.966762,-0.740570,1.469529,-0.962382,1.383525,-1.881596,-0.036612,-2.326094
2,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,0.942118,1.762963,-0.408902,0.514347,-1.126567,1.351461,1.762782,-0.871871,0.768488,...,-1.763825,0.624943,2.821767,1.070980,1.483262,-1.849699,-0.462854,-1.672111,1.660181,1.872878
3,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,-1.875595,0.050555,1.031914,-0.668931,-0.677426,-0.820803,-1.981195,-2.135660,-2.995363,...,-1.222338,-0.192205,-0.901741,1.838929,1.861430,-0.794674,2.664395,1.720787,-0.186903,-1.178385
4,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,-0.974751,2.498883,0.251445,1.301010,0.137949,-0.975547,0.428637,-0.868052,1.202727,...,-2.130194,-1.719221,-0.419598,2.851778,1.108923,-1.456366,-1.579241,0.228512,-0.281135,-1.521479


Loading the Label data

In [12]:

csv_path = "/nfs/home/users/vpandya/tile_level_training_data.csv"
labels_df = pd.read_csv(csv_path)
# labels_df["slide_tile"] = labels_df["slides"].astype(str) + "_" + labels_df["tiles"].astype(str)
print("Labels in CSV:", len(labels_df))
print(labels_df.head())
label = pd.read_csv ("tile_level_training_data.csv")
print ("label loaded successfully")
print("label:", label.shape)
display(label.head())

Labels in CSV: 360567
        samples                   slides       tiles  hpc_id  \
0  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1  18_15.jpeg      32   
1  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1   14_6.jpeg      50   
2  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1  11_23.jpeg      27   
3  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1  10_20.jpeg      21   
4  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1   23_1.jpeg       6   

                           slide_tile malignant_label malignant_subtype  \
0  TCGA-55-7574-01Z-00-DX1_18_15.jpeg              No               NaN   
1   TCGA-55-7574-01Z-00-DX1_14_6.jpeg             Yes             Solid   
2  TCGA-55-7574-01Z-00-DX1_11_23.jpeg             Yes            Acinar   
3  TCGA-55-7574-01Z-00-DX1_10_20.jpeg             Yes             Solid   
4   TCGA-55-7574-01Z-00-DX1_23_1.jpeg              No               NaN   

  second_malignant_subtype    non_malignant_subtype  \
0                      NaN             Collagenosis   
1           Micropapillary      

,samples,slides,tiles,hpc_id,slide_tile,malignant_label,malignant_subtype,second_malignant_subtype,non_malignant_subtype,second_non_malignant_subtype,inflammation_label,necrosis_label,stromal_epithelium_ratio,stromal_cellularity,cluster_homogeneity
0,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,18_15.jpeg,32,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,No,NaN,NaN,Collagenosis,NaN,Marked,None,NaN,NaN,High (> 75%)
1,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,14_6.jpeg,50,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,Yes,Solid,Micropapillary,NaN,NaN,Mild-moderate,None,Roughly equal,Low,High (> 75%)
2,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,11_23.jpeg,27,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,Yes,Acinar,Cribriform,Collagenosis,NaN,None-sparse,None,More stroma,Low,High (> 75%)
3,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,10_20.jpeg,21,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,Yes,Solid,Acinar,NaN,NaN,Mild-moderate,None,Roughly equal,Moderate,High (> 75%)
4,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,23_1.jpeg,6,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,No,NaN,NaN,Vessels (wall or lumen),NaN,None-sparse,None,NaN,NaN,High (> 75%)


In [13]:

# remove the column
label = label.drop(columns=["hpc_id"])

print(label.shape)
display(label.head())

(360567, 14)


,samples,slides,tiles,slide_tile,malignant_label,malignant_subtype,second_malignant_subtype,non_malignant_subtype,second_non_malignant_subtype,inflammation_label,necrosis_label,stromal_epithelium_ratio,stromal_cellularity,cluster_homogeneity
0,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,18_15.jpeg,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,No,NaN,NaN,Collagenosis,NaN,Marked,None,NaN,NaN,High (> 75%)
1,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,14_6.jpeg,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,Yes,Solid,Micropapillary,NaN,NaN,Mild-moderate,None,Roughly equal,Low,High (> 75%)
2,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,11_23.jpeg,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,Yes,Acinar,Cribriform,Collagenosis,NaN,None-sparse,None,More stroma,Low,High (> 75%)
3,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,10_20.jpeg,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,Yes,Solid,Acinar,NaN,NaN,Mild-moderate,None,Roughly equal,Moderate,High (> 75%)
4,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,23_1.jpeg,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,No,NaN,NaN,Vessels (wall or lumen),NaN,None-sparse,None,NaN,NaN,High (> 75%)


In [ ]:
# # Load labeled tile IDs
# labels = pd.read_csv("/nfs/home/users/vpandya/TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv")
# tile_ids = set(labels["tile_id"].astype(str))

# # Prepare output lists
# records = []
# batch_size = 50000  # adjust for memory

# with h5py.File("/data/shared/1 Week/Kai-Rakovic-M42/Vaidehi/hdf5_TCGA_LUAD_5x_he_train_reps.h5", "r") as f:
#     keys = list(f.keys())
#     print(f"Total embeddings in .h5: {len(keys)}")

#     matched = [k for k in keys if k in tile_ids]
#     print(f"Embeddings matching labeled tiles: {len(matched)}")

#     for i, k in enumerate(matched, 1):
#         try:
#             arr = f[k]["z_latent"][:]
#             records.append([k] + arr.tolist())
#         except Exception:
#             continue

#         # periodically write to CSV to avoid RAM overflow
#         if i % batch_size == 0 or i == len(matched):
#             temp = pd.DataFrame(records, columns=["tile_id"] + [f"z_{i}" for i in range(len(arr))])
#             temp.to_csv("train_img_z_latent_partial.csv", mode="a", index=False, header=not bool(i - batch_size))
#             records = []
#             print(f" Processed {i}/{len(matched)} tiles")

# print("Done. All embeddings saved to train_img_z_latent_partial.csv")

Filtering the z_latent data based on the label

In [14]:

# Filter
filtered_z = z_latent[z_latent["slide_tile"].isin(label["slide_tile"])]

print(" Filtered z_latent shape:", filtered_z.shape)

display(filtered_z.head())

z_ids = z_latent[["slide_tile"]]
# print("Labels:", labels.shape)
print("Embeddings (IDs only):", z_ids.shape)

 Filtered z_latent shape: (360683, 129)


,slide_tile,z_0,z_1,z_2,z_3,z_4,z_5,z_6,z_7,z_8,...,z_118,z_119,z_120,z_121,z_122,z_123,z_124,z_125,z_126,z_127
0,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,-0.535568,1.992608,1.974693,1.449399,0.355554,-0.467877,-1.488603,0.936622,-1.195256,...,1.027037,-1.529267,0.688359,1.131204,0.451487,1.851114,-1.451645,1.063994,0.495548,0.291344
1,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,-0.966781,0.355278,0.165919,-0.509434,1.690443,0.703921,-1.080820,-0.774081,1.289157,...,-0.977019,1.144525,-2.966762,-0.740570,1.469529,-0.962382,1.383525,-1.881596,-0.036612,-2.326094
2,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,0.942118,1.762963,-0.408902,0.514347,-1.126567,1.351461,1.762782,-0.871871,0.768488,...,-1.763825,0.624943,2.821767,1.070980,1.483262,-1.849699,-0.462854,-1.672111,1.660181,1.872878
3,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,-1.875595,0.050555,1.031914,-0.668931,-0.677426,-0.820803,-1.981195,-2.135660,-2.995363,...,-1.222338,-0.192205,-0.901741,1.838929,1.861430,-0.794674,2.664395,1.720787,-0.186903,-1.178385
4,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,-0.974751,2.498883,0.251445,1.301010,0.137949,-0.975547,0.428637,-0.868052,1.202727,...,-2.130194,-1.719221,-0.419598,2.851778,1.108923,-1.456366,-1.579241,0.228512,-0.281135,-1.521479


Embeddings (IDs only): (826977, 1)


In [15]:
print("Label file  slide_tile examples:")
print(label["slide_tile"].head(5).tolist())

print("\nEmbedding file slide_tile examples:")
print(pd.read_csv("train_img_z_latent.csv.gz", usecols=["slide_tile"], nrows=5)["slide_tile"].tolist())

Label file  slide_tile examples:
['TCGA-55-7574-01Z-00-DX1_18_15.jpeg', 'TCGA-55-7574-01Z-00-DX1_14_6.jpeg', 'TCGA-55-7574-01Z-00-DX1_11_23.jpeg', 'TCGA-55-7574-01Z-00-DX1_10_20.jpeg', 'TCGA-55-7574-01Z-00-DX1_23_1.jpeg']

Embedding file slide_tile examples:
['TCGA-55-7574-01Z-00-DX1_18_15.jpeg', 'TCGA-55-7574-01Z-00-DX1_14_6.jpeg', 'TCGA-55-7574-01Z-00-DX1_11_23.jpeg', 'TCGA-55-7574-01Z-00-DX1_10_20.jpeg', 'TCGA-55-7574-01Z-00-DX1_23_1.jpeg']


Save the filteres Z_latents to a new .csv

In [16]:

filtered_z.to_csv("train_img_z_latent_final.csv", index=False)
print("Saved the final embeddings to train_img_z_latent_final.csv")
z = pd.read_csv("train_img_z_latent_final.csv")
print("Loaded final embeddings.")
print("Shape:", z.shape)
z.head(100)

Saved the final embeddings to train_img_z_latent_final.csv
Loaded final embeddings.
Shape: (360683, 129)


,slide_tile,z_0,z_1,z_2,z_3,z_4,z_5,z_6,z_7,z_8,...,z_118,z_119,z_120,z_121,z_122,z_123,z_124,z_125,z_126,z_127
0,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,-0.535568,1.992608,1.974693,1.449399,0.355554,-0.467877,-1.488603,0.936622,-1.195256,...,1.027037,-1.529267,0.688359,1.131204,0.451487,1.851114,-1.451645,1.063994,0.495548,0.291344
1,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,-0.966781,0.355278,0.165919,-0.509434,1.690443,0.703921,-1.080820,-0.774081,1.289157,...,-0.977019,1.144525,-2.966762,-0.740570,1.469529,-0.962382,1.383525,-1.881596,-0.036612,-2.326094
2,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,0.942118,1.762963,-0.408902,0.514347,-1.126567,1.351461,1.762782,-0.871871,0.768488,...,-1.763825,0.624943,2.821767,1.070980,1.483262,-1.849699,-0.462854,-1.672111,1.660181,1.872878
3,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,-1.875595,0.050555,1.031914,-0.668931,-0.677426,-0.820803,-1.981195,-2.135660,-2.995363,...,-1.222338,-0.192205,-0.901741,1.838929,1.861430,-0.794674,2.664395,1.720787,-0.186903,-1.178385
4,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,-0.974751,2.498883,0.251445,1.301010,0.137949,-0.975547,0.428637,-0.868052,1.202727,...,-2.130194,-1.719221,-0.419598,2.851778,1.108923,-1.456366,-1.579241,0.228512,-0.281135,-1.521479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TCGA-55-7574-01Z-00-DX1_19_13.jpeg,-0.165019,0.507575,1.488384,0.781838,-1.650388,1.599036,1.685820,-1.486457,2.174102,...,-2.471906,1.914087,0.942384,2.128868,1.494326,-0.912078,-1.005567,-2.467212,2.059118,-1.420495
96,TCGA-55-7574-01Z-00-DX1_11_13.jpeg,-2.235712,1.142330,2.203737,2.168683,1.543088,0.206273,-1.957385,0.200595,-2.322848,...,-1.642437,1.966476,-0.045415,-0.724213,-0.218042,-1.963707,1.797223,-0.928538,1.747346,-0.980004
97,TCGA-55-7574-01Z-00-DX1_12_22.jpeg,1.098467,2.338437,1.214252,-0.709002,-0.505936,1.463567,2.204255,-1.122122,0.666074,...,-1.948513,0.413139,2.559600,1.264969,2.280864,-2.084725,0.468058,-1.111634,2.155142,1.730698
98,TCGA-55-7574-01Z-00-DX1_9_25.jpeg,1.827876,0.650669,1.757941,-0.719754,-0.419641,1.098636,0.742654,-0.956813,-1.783096,...,-1.896499,0.884328,0.853649,1.198733,-0.639453,-0.603124,-0.400703,-0.500282,1.020393,1.371974


In [17]:
set_label = set(label["slide_tile"])
set_z = set(filtered_z["slide_tile"])

print("In labels not in z_latent:", len(set_label - set_z))
print("In z_latent not in labels:", len(set_z - set_label))

#Checking which ones are exactly the problem
merged_check = label.merge(filtered_z, on="slide_tile", how="inner")
dupe_keys = merged_check["slide_tile"][merged_check["slide_tile"].duplicated(keep=False)]
print("Problematic slide_tile IDs:", dupe_keys.unique()[:20])
print("Total problematic keys:", len(dupe_keys.unique()))

In labels not in z_latent: 0
In z_latent not in labels: 0
Problematic slide_tile IDs: ['TCGA-49-4488-01Z-00-DX1_39_8.jpeg' 'TCGA-49-4488-01Z-00-DX1_19_23.jpeg'
 'TCGA-49-4488-01Z-00-DX1_31_24.jpeg' 'TCGA-49-4488-01Z-00-DX1_25_21.jpeg'
 'TCGA-49-4488-01Z-00-DX1_25_14.jpeg' 'TCGA-49-4488-01Z-00-DX1_13_13.jpeg'
 'TCGA-49-4488-01Z-00-DX1_42_11.jpeg' 'TCGA-49-4488-01Z-00-DX1_35_23.jpeg'
 'TCGA-49-4488-01Z-00-DX1_32_23.jpeg' 'TCGA-49-4488-01Z-00-DX1_24_15.jpeg'
 'TCGA-49-4488-01Z-00-DX1_30_21.jpeg' 'TCGA-49-4488-01Z-00-DX1_22_23.jpeg'
 'TCGA-49-4488-01Z-00-DX1_42_10.jpeg' 'TCGA-49-4488-01Z-00-DX1_23_24.jpeg'
 'TCGA-49-4488-01Z-00-DX1_25_24.jpeg' 'TCGA-49-4488-01Z-00-DX1_35_24.jpeg'
 'TCGA-49-4488-01Z-00-DX1_45_11.jpeg' 'TCGA-49-4488-01Z-00-DX1_33_19.jpeg'
 'TCGA-49-4488-01Z-00-DX1_17_17.jpeg' 'TCGA-49-4488-01Z-00-DX1_44_19.jpeg']
Total problematic keys: 116


In [18]:
z_latent_clean = filtered_z.drop_duplicates(subset="slide_tile", keep="first").copy()
print("After removing duplicates:", z_latent_clean.shape)
print("Duplicates left:", z_latent_clean.duplicated(subset='slide_tile').sum())

After removing duplicates: (360567, 129)
Duplicates left: 0


Merging Label and z_latent

In [19]:
merged = label.merge(z_latent_clean, on="slide_tile", how="inner")

print(" Merged dataset shape:", merged.shape)
print("Columns:", merged.columns[:100].tolist(), "...")

# Confirm no missing embeddings
print("Missing embeddings:", merged[[c for c in merged.columns if c.startswith("z_")]].isna().any().sum())

# Confirm each tile appears once
print("Unique tiles:", merged["tiles"].nunique(), " / Total rows:", len(merged))

print("Total rows:", len(merged))
print("Unique slides:", merged["slides"].nunique())
print("Unique samples:", merged["samples"].nunique())
print("Unique slide_tile:", merged["slide_tile"].nunique())

 Merged dataset shape: (360567, 142)
Columns: ['samples', 'slides', 'tiles', 'slide_tile', 'malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'cluster_homogeneity', 'z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8', 'z_9', 'z_10', 'z_11', 'z_12', 'z_13', 'z_14', 'z_15', 'z_16', 'z_17', 'z_18', 'z_19', 'z_20', 'z_21', 'z_22', 'z_23', 'z_24', 'z_25', 'z_26', 'z_27', 'z_28', 'z_29', 'z_30', 'z_31', 'z_32', 'z_33', 'z_34', 'z_35', 'z_36', 'z_37', 'z_38', 'z_39', 'z_40', 'z_41', 'z_42', 'z_43', 'z_44', 'z_45', 'z_46', 'z_47', 'z_48', 'z_49', 'z_50', 'z_51', 'z_52', 'z_53', 'z_54', 'z_55', 'z_56', 'z_57', 'z_58', 'z_59', 'z_60', 'z_61', 'z_62', 'z_63', 'z_64', 'z_65', 'z_66', 'z_67', 'z_68', 'z_69', 'z_70', 'z_71', 'z_72', 'z_73', 'z_74', 'z_75', 'z_76', 'z_77', 'z_78', 'z_79', 'z_80', 'z_81', 'z_82', 'z_83', 'z_84', 

In [20]:
dupes = label[label["slide_tile"].duplicated(keep=False)]
print("Duplicate rows:", len(dupes))
dupes_sorted = dupes.sort_values("slide_tile")
dupes_sorted.head(100)

display(merged.head(100))

Duplicate rows: 0


,samples,slides,tiles,slide_tile,malignant_label,malignant_subtype,second_malignant_subtype,non_malignant_subtype,second_non_malignant_subtype,inflammation_label,...,z_118,z_119,z_120,z_121,z_122,z_123,z_124,z_125,z_126,z_127
0,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,18_15.jpeg,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,No,NaN,NaN,Collagenosis,NaN,Marked,...,1.027037,-1.529267,0.688359,1.131204,0.451487,1.851114,-1.451645,1.063994,0.495548,0.291344
1,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,14_6.jpeg,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,Yes,Solid,Micropapillary,NaN,NaN,Mild-moderate,...,-0.977019,1.144525,-2.966762,-0.740570,1.469529,-0.962382,1.383525,-1.881596,-0.036612,-2.326094
2,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,11_23.jpeg,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,Yes,Acinar,Cribriform,Collagenosis,NaN,None-sparse,...,-1.763825,0.624943,2.821767,1.070980,1.483262,-1.849699,-0.462854,-1.672111,1.660181,1.872878
3,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,10_20.jpeg,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,Yes,Solid,Acinar,NaN,NaN,Mild-moderate,...,-1.222338,-0.192205,-0.901741,1.838929,1.861430,-0.794674,2.664395,1.720787,-0.186903,-1.178385
4,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,23_1.jpeg,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,No,NaN,NaN,Vessels (wall or lumen),NaN,None-sparse,...,-2.130194,-1.719221,-0.419598,2.851778,1.108923,-1.456366,-1.579241,0.228512,-0.281135,-1.521479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,19_13.jpeg,TCGA-55-7574-01Z-00-DX1_19_13.jpeg,No,NaN,NaN,Collagenosis,Vessels (wall or lumen),None-sparse,...,-2.471906,1.914087,0.942384,2.128868,1.494326,-0.912078,-1.005567,-2.467212,2.059118,-1.420495
96,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,11_13.jpeg,TCGA-55-7574-01Z-00-DX1_11_13.jpeg,Yes,Solid,Acinar,NaN,NaN,Mild-moderate,...,-1.642437,1.966476,-0.045415,-0.724213,-0.218042,-1.963707,1.797223,-0.928538,1.747346,-0.980004
97,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,12_22.jpeg,TCGA-55-7574-01Z-00-DX1_12_22.jpeg,Yes,Acinar,Cribriform,Collagenosis,NaN,None-sparse,...,-1.948513,0.413139,2.559600,1.264969,2.280864,-2.084725,0.468058,-1.111634,2.155142,1.730698
98,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,9_25.jpeg,TCGA-55-7574-01Z-00-DX1_9_25.jpeg,No,NaN,NaN,Collagenosis,Vessels (wall or lumen),None-sparse,...,-1.896499,0.884328,0.853649,1.198733,-0.639453,-0.603124,-0.400703,-0.500282,1.020393,1.371974


In [21]:

dupes = merged[merged.duplicated(subset="slide_tile", keep=False)]
print("Duplicate slide_tile rows:", len(dupes))

identical = 0
for slide_tile, g in dupes.groupby("slide_tile"):
    vals = g[[c for c in merged.columns if c.startswith("z_")]].astype(float).values
    if len(vals) > 1:
        # Compare all rows in this group to the first one
        if all(np.allclose(vals[0], v, atol=1e-6, rtol=1e-6) for v in vals[1:]):
            identical += 1

print(f"Identical duplicate embeddings: {identical}")

Duplicate slide_tile rows: 0
Identical duplicate embeddings: 0


In [ ]:
# meta_cols = ["samples", "slides", "tiles", "slide_tile"]

# merged_clean = merged.drop(columns=meta_cols)
# print("Old shape:", merged.shape)
# print("New shape:", merged_clean.shape)
# print("Columns now:", merged_clean.columns.tolist()[:20])

Old shape: (360567, 142)
New shape: (360567, 138)
Columns now: ['malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'cluster_homogeneity', 'z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8', 'z_9']


In [22]:
merged.to_csv("training_data.csv", index=False)

In [ ]:
# # from sklearn.model_selection import GroupShuffleSplit

# groups = merged["samples"]

# # first split train vs temp (val+test)
# gss1 = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
# train_idx, temp_idx = next(gss1.split(merged, merged["malignant_label"], groups))

# train_df = merged.iloc[train_idx]
# temp_df = merged.iloc[temp_idx]

# # second split val vs test
# gss2 = GroupShuffleSplit(test_size=0.5, n_splits=1, random_state=42)
# val_idx, test_idx = next(gss2.split(temp_df, temp_df["malignant_label"], temp_df["samples"]))

# val_df = temp_df.iloc[val_idx]
# test_df = temp_df.iloc[test_idx]

# print("Train samples:", train_df['samples'].nunique())
# print("Val samples:", val_df['samples'].nunique())
# print("Test samples:", test_df['samples'].nunique())

# for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
#     print(f"\n{name} malignant ratio:")
#     print(df["malignant_label"].value_counts(normalize=True))

# for label in ["inflammation_label", "necrosis_label", "cluster_homogeneity"]:
#     print(f"\nLabel: {label}")
#     print("Train:", train_df[label].nunique(), 
#           "Val:", val_df[label].nunique(), 
#           "Test:", test_df[label].nunique())

# for col in ["malignant_label", "inflammation_label", "necrosis_label"]:
#     print(col, ":", 
#           "Train:", train_df[col].nunique(), 
#           "Val:", val_df[col].nunique(), 
#           "Test:", test_df[col].nunique())
    
# feature_cols = [c for c in train_df.columns if c.startswith("z_")]
# print("NaNs in train:", train_df[feature_cols].isna().sum().sum())
# print("NaNs in val:", val_df[feature_cols].isna().sum().sum())
# print("NaNs in test:", test_df[feature_cols].isna().sum().sum())

# set(train_df["samples"]) & set(val_df["samples"]) & set(test_df["samples"])

Train samples: 307
Val samples: 66
Test samples: 66

Train malignant ratio:
malignant_label
Yes    0.55395
No     0.44605
Name: proportion, dtype: float64

Val malignant ratio:
malignant_label
Yes    0.566079
No     0.433921
Name: proportion, dtype: float64

Test malignant ratio:
malignant_label
Yes    0.568812
No     0.431188
Name: proportion, dtype: float64

Label: inflammation_label
Train: 3 Val: 3 Test: 3

Label: necrosis_label
Train: 3 Val: 3 Test: 3

Label: cluster_homogeneity
Train: 2 Val: 2 Test: 2
malignant_label : Train: 2 Val: 2 Test: 2
inflammation_label : Train: 3 Val: 3 Test: 3
necrosis_label : Train: 3 Val: 3 Test: 3
NaNs in train: 0
NaNs in val: 0
NaNs in test: 0


set()

In [ ]:
# train_df.to_csv("train_split.csv", index=False)
# val_df.to_csv("val_split.csv", index=False)
# test_df.to_csv("test_split.csv", index=False)
# print("Saved train/val/test splits successfully.")

Saved train/val/test splits successfully.


In [ ]:
# df_train = pd.read_csv("train_split.csv")
# df_val = pd.read_csv("val_split.csv")
# df_test = pd.read_csv("test_split.csv")

In [ ]:
# display(df_train)

,samples,slides,tiles,hpc_id,slide_tile,malignant_label,malignant_subtype,second_malignant_subtype,non_malignant_subtype,second_non_malignant_subtype,...,z_118,z_119,z_120,z_121,z_122,z_123,z_124,z_125,z_126,z_127
0,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,18_15.jpeg,32,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,No,NaN,NaN,Collagenosis,NaN,...,1.027037,-1.529267,0.688359,1.131204,0.451487,1.851114,-1.451645,1.063994,0.495548,0.291344
1,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,14_6.jpeg,50,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,Yes,Solid,Micropapillary,NaN,NaN,...,-0.977019,1.144525,-2.966762,-0.740570,1.469529,-0.962382,1.383525,-1.881596,-0.036612,-2.326094
2,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,11_23.jpeg,27,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,Yes,Acinar,Cribriform,Collagenosis,NaN,...,-1.763825,0.624943,2.821767,1.070980,1.483262,-1.849699,-0.462854,-1.672111,1.660181,1.872878
3,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,10_20.jpeg,21,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,Yes,Solid,Acinar,NaN,NaN,...,-1.222338,-0.192205,-0.901741,1.838929,1.861430,-0.794674,2.664395,1.720787,-0.186903,-1.178385
4,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,23_1.jpeg,6,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,No,NaN,NaN,Vessels (wall or lumen),NaN,...,-2.130194,-1.719221,-0.419598,2.851778,1.108923,-1.456366,-1.579241,0.228512,-0.281135,-1.521479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251554,TCGA-86-8669,TCGA-86-8669-01Z-00-DX1,19_17.jpeg,54,TCGA-86-8669-01Z-00-DX1_19_17.jpeg,Yes,Papillary,Micropapillary,NaN,NaN,...,2.937278,0.002108,0.396947,0.527721,-1.828786,-0.067616,1.013980,-1.489775,1.735244,-1.728110
251555,TCGA-86-8669,TCGA-86-8669-01Z-00-DX1,27_13.jpeg,11,TCGA-86-8669-01Z-00-DX1_27_13.jpeg,Yes,Acinar,Papillary,NaN,NaN,...,0.585684,0.563123,2.529112,-0.482992,-0.946280,-0.287938,2.512579,0.167921,-0.097062,-0.746370
251556,TCGA-86-8669,TCGA-86-8669-01Z-00-DX1,9_5.jpeg,54,TCGA-86-8669-01Z-00-DX1_9_5.jpeg,Yes,Papillary,Micropapillary,NaN,NaN,...,2.058870,0.495218,-1.044787,0.242564,-2.331643,-1.021795,-0.617157,-1.577793,2.210438,-2.411570
251557,TCGA-86-8669,TCGA-86-8669-01Z-00-DX1,35_5.jpeg,17,TCGA-86-8669-01Z-00-DX1_35_5.jpeg,Yes,Solid,NaN,NaN,NaN,...,2.537841,0.680686,-0.936299,1.603511,0.295326,-2.671169,0.384753,0.245438,1.368378,0.529594


In [ ]:
# meta_cols = ['samples', 'slides', 'tiles', 'slide_tile']
# meta_train = df_train[meta_cols].copy()
# meta_val   = df_val[meta_cols].copy()
# meta_test  = df_test[meta_cols].copy()

In [ ]:
# df_train = df_train.drop(columns=meta_cols)
# df_val   = df_val.drop(columns=meta_cols)
# df_test  = df_test.drop(columns=meta_cols)

In [ ]:
# feature_cols = [c for c in df_train.columns if c.startswith('z_')]
# label_cols = [c for c in df_train.columns if not c.startswith('z_')]

In [ ]:
# X_train = df_train[feature_cols].values
# X_val   = df_val[feature_cols].values
# X_test  = df_test[feature_cols].values
# Y_train = df_train[label_cols]
# Y_val   = df_val[label_cols]
# Y_test  = df_test[label_cols]

In [ ]:
# from sklearn.preprocessing import LabelEncoder

# # Fill NaNs for conditional columns
# Y_train = Y_train.fillna('None')
# Y_val   = Y_val.fillna('None')
# Y_test  = Y_test.fillna('None')

# # Define hierarchical label columns
# malignant_branch = [
#     'stromal_epithelium_ratio',
#     'stromal_cellularity',
#     'malignant_subtype',
#     'second_malignant_subtype'
# ]
# non_malignant_branch = [
#     'non_malignant_subtype',
#     'second_non_malignant_subtype'
# ]
# top_level = [
#     'cluster_homogeneity',
#     'malignant_label',
#     'inflammation_label',
#     'necrosis_label'
# ]

# all_label_cols = top_level + malignant_branch + non_malignant_branch

# encoders = {}
# for col in all_label_cols:
#     le = LabelEncoder()
#     le.fit(Y_train[col])
#     encoders[col] = le

#     Y_train[col] = le.transform(Y_train[col])
#     Y_val[col]   = le.transform(Y_val[col])
#     Y_test[col]  = le.transform(Y_test[col])

In [ ]:
# import numpy as np
# import pickle

# # Save embeddings
# np.save("X_train.npy", X_train)
# np.save("X_val.npy",   X_val)
# np.save("X_test.npy",  X_test)

# # Save labels
# Y_train.to_csv("Y_train_encoded.csv", index=False)
# Y_val.to_csv("Y_val_encoded.csv", index=False)
# Y_test.to_csv("Y_test_encoded.csv", index=False)

# # Save label encoders for decoding predictions later
# with open("label_encoders.pkl", "wb") as f:
#     pickle.dump(encoders, f)

# print("All encoded datasets and encoders saved successfully.")

All encoded datasets and encoders saved successfully.


In [1]:
import pandas as pd
df = pd.read_csv("/Users/vaidehipandya/Desktop/training_data.csv")
df.columns.tolist()

['samples',
 'slides',
 'tiles',
 'hpc_id',
 'slide_tile',
 'malignant_label',
 'malignant_subtype',
 'second_malignant_subtype',
 'non_malignant_subtype',
 'second_non_malignant_subtype',
 'inflammation_label',
 'necrosis_label',
 'stromal_epithelium_ratio',
 'stromal_cellularity',
 'cluster_homogeneity',
 'z_0',
 'z_1',
 'z_2',
 'z_3',
 'z_4',
 'z_5',
 'z_6',
 'z_7',
 'z_8',
 'z_9',
 'z_10',
 'z_11',
 'z_12',
 'z_13',
 'z_14',
 'z_15',
 'z_16',
 'z_17',
 'z_18',
 'z_19',
 'z_20',
 'z_21',
 'z_22',
 'z_23',
 'z_24',
 'z_25',
 'z_26',
 'z_27',
 'z_28',
 'z_29',
 'z_30',
 'z_31',
 'z_32',
 'z_33',
 'z_34',
 'z_35',
 'z_36',
 'z_37',
 'z_38',
 'z_39',
 'z_40',
 'z_41',
 'z_42',
 'z_43',
 'z_44',
 'z_45',
 'z_46',
 'z_47',
 'z_48',
 'z_49',
 'z_50',
 'z_51',
 'z_52',
 'z_53',
 'z_54',
 'z_55',
 'z_56',
 'z_57',
 'z_58',
 'z_59',
 'z_60',
 'z_61',
 'z_62',
 'z_63',
 'z_64',
 'z_65',
 'z_66',
 'z_67',
 'z_68',
 'z_69',
 'z_70',
 'z_71',
 'z_72',
 'z_73',
 'z_74',
 'z_75',
 'z_76',
 'z_77',


In [3]:
# KEEP ONLY THESE (adjust column names if needed)
keep_cols = ["slide_tile"]   # or ["slide_id", "tile_id"]
# Identify all z_latent columns
z_cols = [c for c in df.columns if c.startswith("z_")]

# Keep only these
df_clean = df[keep_cols + z_cols].copy()

# Save cleaned file
out_path = "/Users/vaidehipandya/Desktop/Work/training_data_embeddings_only.csv"
df_clean.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Columns kept:", df_clean.columns.tolist())
print(df_clean.head())

Saved: /Users/vaidehipandya/Desktop/Work/training_data_embeddings_only.csv
Columns kept: ['slide_tile', 'z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8', 'z_9', 'z_10', 'z_11', 'z_12', 'z_13', 'z_14', 'z_15', 'z_16', 'z_17', 'z_18', 'z_19', 'z_20', 'z_21', 'z_22', 'z_23', 'z_24', 'z_25', 'z_26', 'z_27', 'z_28', 'z_29', 'z_30', 'z_31', 'z_32', 'z_33', 'z_34', 'z_35', 'z_36', 'z_37', 'z_38', 'z_39', 'z_40', 'z_41', 'z_42', 'z_43', 'z_44', 'z_45', 'z_46', 'z_47', 'z_48', 'z_49', 'z_50', 'z_51', 'z_52', 'z_53', 'z_54', 'z_55', 'z_56', 'z_57', 'z_58', 'z_59', 'z_60', 'z_61', 'z_62', 'z_63', 'z_64', 'z_65', 'z_66', 'z_67', 'z_68', 'z_69', 'z_70', 'z_71', 'z_72', 'z_73', 'z_74', 'z_75', 'z_76', 'z_77', 'z_78', 'z_79', 'z_80', 'z_81', 'z_82', 'z_83', 'z_84', 'z_85', 'z_86', 'z_87', 'z_88', 'z_89', 'z_90', 'z_91', 'z_92', 'z_93', 'z_94', 'z_95', 'z_96', 'z_97', 'z_98', 'z_99', 'z_100', 'z_101', 'z_102', 'z_103', 'z_104', 'z_105', 'z_106', 'z_107', 'z_108', 'z_109', 'z_110', 'z_111'